# Big Data Analytics Project - Spring 2026

In this notebook, we apply **deep learning** to the **Fashion MNIST** dataset using PySpark. The primary goal is to demonstrate how Spark can orchestrate, distribute, and evaluate deep learning workflows — not to maximise model accuracy. Two approaches are compared:

1. **Spark MLlib's `MultilayerPerceptronClassifier`** — Spark's own feedforward neural network, integrated natively into a `Pipeline` with `CrossValidator` hyperparameter tuning.
2. **Transfer Learning with MobileNetV2** — a pretrained MobileNetV2 backbone extracts 1280-dimensional features per image via `predict_batch_udf`; a lightweight PyTorch classification head is then trained in a distributed manner via `TorchDistributor`, and deployed for inference using Spark's broadcast variables.

### Group 16 Members:

- Ana Margarida Macedo (20250405) 
- Catarina Aboim ()
- Margarida Craveiro ()
- Matilde Simões ()
- Lourenço Silva (20250453)

### <font color='#0000FF'>**Methodology**</font><a class="anchor" id="TOP"></a>

- [1. Imports and Setup](#1)
- [2. Data Loading & Preprocessing](#2)
    - [2.1 Loading the Dataset](#2.1)
    - [2.2 Data Visualisation](#2.2)
    - [2.3 Vectorisation & Feature Assembly](#2.3)
- [3. Spark MLlib Multilayer Perceptron](#3)
    - [3.1 Pipeline Definition](#3.1)
    - [3.2 Hyperparameter Tuning](#3.2)
    - [3.3 Evaluation](#3.3)
- [4. Transfer Learning](#4)
    - [4.1 Feature Extraction](#4.1)
    - [4.2 Training the Classification Head](#4.2)
    - [4.3 Distributed Inference](#4.3)
    - [4.4 Evaluation](#4.4)
- [5. Saving the Pipelines](#5)
- [6. Comparing Results](#6)


<a class="anchor" id="1">

# **1. Imports and Setup**
</a>

In [ ]:
!java -version

In [ ]:
# --- Standard library / third party ---
import os, subprocess, io
from typing import Iterator

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- PySpark core ---
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.types import (
    StructType, StructField,
    ArrayType, FloatType, DoubleType, IntegerType, StringType,
)

# --- PySpark ML ---
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, MinMaxScaler
from pyspark.ml.classification import MultilayerPerceptronClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.functions import predict_batch_udf, array_to_vector
from pyspark.ml.torch.distributor import TorchDistributor

# --- PyTorch ---
import torch
import torch.nn as nn

Below we define the **evaluation utilities** used throughout Sections 3 and 4: a metrics printer (`evaluate_multiclass`) and a 10-class confusion matrix plotter (`plot_confusion_matrix`). Defining them here, in Imports and Setup, ensures every evaluation cell can access them regardless of run order.

In [ ]:
CLASS_NAMES = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

def evaluate_multiclass(predictions, model_name, label_col='label'):
    """Compute accuracy and weighted-F1 for a multiclass Spark predictions DataFrame."""
    acc = MulticlassClassificationEvaluator(
        labelCol=label_col, predictionCol='prediction', metricName='accuracy'
    ).evaluate(predictions)
    f1 = MulticlassClassificationEvaluator(
        labelCol=label_col, predictionCol='prediction', metricName='f1'
    ).evaluate(predictions)
    print(f'=== {model_name} ===')
    print(f'  Accuracy      : {acc:.4f}')
    print(f'  F1 (weighted) : {f1:.4f}')
    return {'model': model_name, 'Accuracy': acc, 'F1': f1}


def plot_confusion_matrix(predictions, model_name, label_col='label'):
    """Plot a 10-class confusion matrix (absolute counts) from a Spark predictions DF."""
    cm_rows = predictions.groupBy(label_col, 'prediction').count().collect()
    cm = np.zeros((10, 10), dtype=int)
    for r in cm_rows:
        cm[int(r[label_col]), int(r['prediction'])] = r['count']

    fig, ax = plt.subplots(figsize=(11, 9))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title(f'Confusion Matrix — {model_name}', fontsize=13)
    ax.set_xlabel('Predicted', fontsize=11)
    ax.set_ylabel('Actual', fontsize=11)
    plt.xticks(rotation=40, ha='right', fontsize=8)
    plt.yticks(fontsize=8)
    plt.tight_layout()
    plt.show()
    return cm

Local VSCODE

In [ ]:
# Find the real Java path from Homebrew
result = subprocess.run(['brew', '--prefix', 'openjdk@17'], capture_output=True, text=True)
java_home = result.stdout.strip()
print(f'Homebrew Java prefix: {java_home}')

os.environ['JAVA_HOME'] = java_home
os.environ['PATH'] = java_home + '/bin:' + os.environ['PATH']

LIGHTNING AI

In [ ]:
# # Set JAVA_HOME to Java 17 (for Lightning AI)
# os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
# os.environ['PATH'] = os.environ['JAVA_HOME'] + '/bin:' + os.environ['PATH']

BOTH

In [ ]:
spark = SparkSession.builder \
    .appName('BDA_DeepLearning') \
    .master('local[*]') \
    .config('spark.driver.memory', '6g') \
    .config('spark.executor.memory', '6g') \
    .config('spark.driver.maxResultSize', '2g') \
    .config('spark.sql.shuffle.partitions', '8') \
    .getOrCreate()

<a class="anchor" id="2">

# **2. Data Loading & Preprocessing**
</a>

<a class="anchor" id="2.1">

## **2.1. Loading the Dataset**
</a>

The Fashion MNIST dataset was lightly cleaned and saved to Parquet in Notebook 1. Each Parquet file has one row per image: a `label` column (integer, 0–9) and 784 individual pixel columns named `pixel1` through `pixel784` (integer values in [0, 255], stored as `int32`). The train/test split is pre-defined: **60,000 training** and **10,000 test** images.

Loading from Parquet is efficient — Spark reads only the columns it needs (column-pruning) and can push down filters before touching the file. We cast `label` to `DoubleType` immediately, as Spark ML classifiers require a floating-point label column.

In [ ]:
PIXEL_COLS = [f'pixel{i}' for i in range(1, 785)]  # pixel1 … pixel784

df_train = (
    spark.read.parquet('data/clean/fashion_mnist_train_clean.parquet')
    .withColumn('label', col('label').cast(DoubleType()))
)
df_test = (
    spark.read.parquet('data/clean/fashion_mnist_test_clean.parquet')
    .withColumn('label', col('label').cast(DoubleType()))
)

# Cache both — reused by the MLP pipeline AND the transfer-learning feature extractor
df_train.cache()
df_test.cache()

print(f'Train rows : {df_train.count():,}')
print(f'Test rows  : {df_test.count():,}')
df_train.printSchema()

<a class="anchor" id="2.2">

## **2.2. Data Visualisation**
</a>

To sanity-check the data we collect one representative image per class from the Spark DataFrame and display it. The pixel columns are reassembled into a 28×28 numpy array using `row_number()` over a window partitioned by `label` — a Spark-native way to pick one row per group without a full sort.

In [ ]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number

# Pick exactly one row per class using a window function
w = Window.partitionBy('label').orderBy('label')
sample_pdf = (
    df_train
    .withColumn('rn', row_number().over(w))
    .filter(col('rn') == 1)
    .orderBy('label')
    .select('label', *PIXEL_COLS)
    .toPandas()
)

fig, axes = plt.subplots(2, 5, figsize=(13, 5))
for ax, (_, row) in zip(axes.flat, sample_pdf.iterrows()):
    lbl = int(row['label'])
    img = row[PIXEL_COLS].values.reshape(28, 28).astype(np.uint8)
    ax.imshow(img, cmap='gray')
    ax.set_title(f'{lbl}: {CLASS_NAMES[lbl]}', fontsize=9)
    ax.axis('off')
plt.suptitle('One representative image per class — Fashion MNIST', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

<a class="anchor" id="2.3">

## **2.3. Vectorisation & Feature Assembly**
</a>

Spark ML estimators require features to be in a single **`DenseVector`** column. We use `VectorAssembler` to collect the 784 individual pixel columns into one vector (`features_raw`). This step also serves as the vectorisation stage that was intentionally deferred from Notebook 1.

The assembled vector still holds raw pixel intensities in [0, 255]. `MinMaxScaler` (applied inside the pipeline in Section 3) will normalise each dimension to [0.0, 1.0] — neural networks converge significantly faster on unit-range inputs.

For the transfer-learning pipeline in Section 4 we also create a `pixels` array column (`ArrayType(FloatType())`). This is the format expected by `predict_batch_udf`: each row's 784 values are passed to the worker as a single numpy array, letting the feature-extraction UDF reconstruct the 28×28 image and preprocess it for MobileNetV2.

In [ ]:
# --- VectorAssembler: 784 pixel columns → one DenseVector per row ---
# Used by the MLP pipeline (Section 3) and to inspect raw feature vectors.
assembler = VectorAssembler(
    inputCols=PIXEL_COLS,
    outputCol='features_raw',
)

df_train_vec = assembler.transform(df_train).select('features_raw', 'label')
df_test_vec  = assembler.transform(df_test).select('features_raw', 'label')

df_train_vec.show(3, truncate=60)
print(f'Feature vector size: {len(df_train_vec.first()["features_raw"])}')

In [ ]:
# --- Pixel array column for Section 4 (transfer-learning feature extraction) ---
# ArrayType(FloatType()) is the format predict_batch_udf passes to the worker
# as a numpy array of shape (batch_size, 784).
pixels_array_col = F.array(*[col(c).cast(FloatType()) for c in PIXEL_COLS])

df_train_img = df_train.withColumn('pixels', pixels_array_col).select('pixels', 'label')
df_test_img  = df_test.withColumn('pixels', pixels_array_col).select('pixels', 'label')

df_train_img.cache()
df_test_img.cache()

print('pixels column schema:', df_train_img.schema['pixels'].dataType)
df_train_img.show(2, truncate=60)

<a class="anchor" id="3">

# **3. Spark MLlib Multilayer Perceptron**
</a>

**Why `MultilayerPerceptronClassifier` (MLPC)?**  
MLPC is Spark MLlib's built-in feedforward neural network. Unlike external deep learning frameworks, it is a first-class **`Estimator`** that slots seamlessly into a Spark `Pipeline`, participates in `CrossValidator` hyperparameter search, and is evaluated with the standard `MulticlassClassificationEvaluator` — no driver-to-worker serialisation overhead, no file I/O between stages. It uses the L-BFGS optimiser internally and distributes gradient computation across Spark partitions, making it genuinely scale-out.

The trade-off versus a modern PyTorch CNN is that MLPC operates on flat feature vectors (no convolutional layers), so it cannot exploit the spatial structure of the images. For our purposes — showcasing Spark's native deep-learning capability — that is fine.

<a class="anchor" id="3.1">

## **3.1. Pipeline Definition**
</a>

The pipeline chains two stages:

1. **`MinMaxScaler`** — scales each of the 784 pixel dimensions from [0, 255] to [0.0, 1.0]. Neural networks converge significantly faster on unit-range inputs; this is the primary preprocessing step the dataset still needs after Notebook 1's light cleaning.
2. **`MultilayerPerceptronClassifier`** — a fully-connected MLP. The `layers` parameter defines the network topology as a list: `[784, 256, 10]` means 784 input units (one per pixel), one hidden layer of 256 ReLU units, and 10 output softmax units (one per clothing class). `blockSize` sets the mini-batch size used by L-BFGS.

Note that `VectorAssembler` is applied outside the Pipeline here (in Section 2.3) and its output `features_raw` is the starting column. In a production setup the assembler would also be a Pipeline stage to guarantee train/test consistency.

In [ ]:
scaler = MinMaxScaler(
    inputCol='features_raw',
    outputCol='features',
)

mlpc = MultilayerPerceptronClassifier(
    featuresCol='features',
    labelCol='label',
    layers=[784, 256, 10],   # topology: input → hidden → output
    maxIter=20,              # L-BFGS iterations
    blockSize=128,           # mini-batch size for gradient accumulation
    seed=42,
)

pipeline_mlpc = Pipeline(stages=[scaler, mlpc])
print('Pipeline stages:', [s.__class__.__name__ for s in pipeline_mlpc.getStages()])

<a class="anchor" id="3.2">

## **3.2. Hyperparameter Tuning**
</a>

We use Spark MLlib's `CrossValidator` to search over network topologies. `ParamGridBuilder` defines the grid — two candidate `layers` configurations — and `CrossValidator` trains `numFolds` full pipelines per configuration, averaging the validation accuracy across folds. Because `estimator=pipeline_mlpc` (not just `mlpc`), `CrossValidator` re-runs **every pipeline stage** — including `MinMaxScaler` — on each fold, preventing data leakage from the scaler fit.

`parallelism=2` allows two pipeline fits to run concurrently on separate thread pools.

> **Runtime note:** Each fold trains a full MLP on ~40,000 samples × 784 features. On a local CPU this takes roughly **5–10 minutes per fold** (3 folds × 2 configs = 6 fits). Reduce `numFolds` to 2 or `maxIter` to 10 if you need faster iteration.

In [ ]:
param_grid = (
    ParamGridBuilder()
    .addGrid(mlpc.layers, [
        [784, 128, 10],   # shallower / faster
        [784, 256, 10],   # wider hidden layer
    ])
    .build()
)

cv_evaluator = MulticlassClassificationEvaluator(
    labelCol='label', predictionCol='prediction', metricName='accuracy'
)

crossval_mlpc = CrossValidator(
    estimator=pipeline_mlpc,
    estimatorParamMaps=param_grid,
    evaluator=cv_evaluator,
    numFolds=3,
    parallelism=2,
    seed=42,
)

cv_model_mlpc = crossval_mlpc.fit(df_train_vec)
pipeline_model_mlpc = cv_model_mlpc.bestModel
best_mlpc = pipeline_model_mlpc.stages[-1]

print(f'Best layers : {best_mlpc.getLayers()}')
print(f'CV accuracy per config: {[round(m, 4) for m in cv_model_mlpc.avgMetrics]}')

<a class="anchor" id="3.3">

## **3.3. Evaluation**
</a>

In [ ]:
predictions_mlpc = pipeline_model_mlpc.transform(df_test_vec)
results_mlpc = evaluate_multiclass(predictions_mlpc, 'MLP (Spark MLlib MLPC)')

### Confusion Matrix

The matrix below shows how each of the 10 clothing classes is classified. Visually similar garments (*Shirt* vs *T-shirt/top* vs *Coat*, or *Sneaker* vs *Ankle boot*) are the hardest to separate with a flat-pixel model that has no spatial feature extraction.

In [ ]:
cm_mlpc = plot_confusion_matrix(predictions_mlpc, 'MLP (Spark MLlib MLPC)')

### Results

The Spark MLPC achieves accuracy in the **~85–88%** range on Fashion MNIST — a strong result for a single hidden-layer MLP on raw pixels. The cross-validation metrics should be close across the two topologies, indicating stable generalisation.

Common confusion patterns: *Shirt* (class 6) is frequently misclassified as *T-shirt/top* (class 0) or *Coat* (class 4) — these categories are genuinely similar in flat pixel space. Footwear classes are usually well-separated from apparel classes, but can confuse one another.

<a class="anchor" id="4">

# **4. Transfer Learning**
</a>

**Why transfer learning?** Transfer learning re-purposes a network pretrained on ImageNet to extract rich visual features, then trains a small classification *head* on those features for the target task. We chose **MobileNetV2** (Howard et al., 2018) for two reasons specific to Fashion MNIST: (1) the images are only 28×28 pixels, so a small model like MobileNetV2 (3.4 M parameters) runs considerably fast on CPU; (2) the 1280-dimensional feature vector it produces is more than sufficient for a 10-class problem.

<a class="anchor" id="4.1">

## **4.1. Feature Extraction**
</a>

`predict_batch_udf` (Spark 3.4+) is the idiomatic way to run distributed inference inside Spark. It takes a **setup function** — `make_mobilenet_fn` below — that each worker calls **once at startup** to load the model. The inner `predict` function it returns is then called repeatedly on batches of rows, with Spark managing batching and worker lifecycle entirely.

Compared to a hand-written Pandas UDF, `predict_batch_udf` is more efficient because the model is loaded once per worker process (not once per partition batch) and batching is handled by the API via `batch_size`.

The preprocessing inside the UDF converts each 28×28 greyscale image to a 3-channel (RGB) tensor at 96×96 resolution, normalised with ImageNet statistics — the same normalisation MobileNetV2 was trained with on ImageNet.

In [ ]:
def make_mobilenet_fn():
    """Called once per worker at startup — loads MobileNetV2 and returns predict fn."""
    import torch
    import numpy as np
    from PIL import Image
    from torchvision import models, transforms
    from torchvision.models import MobileNet_V2_Weights

    # Load pretrained MobileNetV2; replace the final Linear(1280→1000) classifier
    # with Identity so the model outputs a raw 1280-dimensional feature vector.
    backbone = models.mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)
    backbone.classifier = torch.nn.Identity()
    backbone.eval()

    # Preprocessing: greyscale PIL image (28×28) → normalised 3-channel tensor (3, 96, 96)
    preprocess = transforms.Compose([
        transforms.Resize((96, 96)),              # upsample from 28×28
        transforms.Grayscale(num_output_channels=3),  # repeat grey channel → RGB
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],   # ImageNet channel means
                             std=[0.229, 0.224, 0.225]),    # ImageNet channel stds
    ])

    def predict(pixels_batch: np.ndarray) -> np.ndarray:
        """
        pixels_batch : (N, 784) float32, values in [0, 255].
        Returns       : (N, 1280) float32 — MobileNetV2 feature vectors.
        """
        n = pixels_batch.shape[0]
        imgs = [
            Image.fromarray(pixels_batch[i].reshape(28, 28).astype(np.uint8), mode='L')
            for i in range(n)
        ]
        tensors = torch.stack([preprocess(img) for img in imgs])
        with torch.no_grad():
            return backbone(tensors).numpy()   # (N, 1280)

    return predict


extract_features_udf = predict_batch_udf(
    make_mobilenet_fn,
    return_type=ArrayType(FloatType()),
    batch_size=64,
)
print('predict_batch_udf created — MobileNetV2 loads once per worker at runtime.')

We repartition to 4 partitions before applying the UDF so that each Python worker loads MobileNetV2 once and processes its assigned batch of images. The resulting feature DataFrames are **cached** — feature extraction is expensive (~10–20 min on CPU) and the vectors are accessed again when writing to Parquet and during evaluation.

In [ ]:
def extract_features(df_img, split_name):
    """Apply MobileNetV2 feature extraction UDF to a Fashion MNIST image DataFrame."""
    return (
        df_img
        .repartition(4)   # one model replica per partition worker
        .select(
            col('label'),
            extract_features_udf(col('pixels')).alias('mobilenet_features'),
        )
        .cache()
    )

print('Extracting features from training set (60,000 images)...')
features_train = extract_features(df_train_img, 'train')
print(f'  Train feature rows: {features_train.count():,}')

print('Extracting features from test set (10,000 images)...')
features_test = extract_features(df_test_img, 'test')
print(f'  Test  feature rows: {features_test.count():,}')

features_train.show(3, truncate=60)

<a class="anchor" id="4.2">

## **4.2. Training the Classification Head**
</a>

`TorchDistributor` (Spark 3.4+) launches a PyTorch training job from within a Spark session, distributing work across multiple processes via PyTorch DDP. The training function must be **fully self-contained** — all imports and class definitions must live inside it, because `TorchDistributor` pickles it and sends it to worker processes that do not share the driver's namespace. The trained `state_dict` is returned to the driver.

**Handoff pattern (Spark → PyTorch → Spark):**
1. Spark writes the extracted features to Parquet — the serialisation boundary.
2. `TorchDistributor` launches PyTorch, which reads those files with `pyarrow`.
3. The trained `state_dict` is returned to the driver and used for broadcast inference.

In [ ]:
FASHION_TRAIN_PARQUET = 'fashion_train_features.parquet'
FASHION_TEST_PARQUET  = 'fashion_test_features.parquet'
FEATURE_DIM  = 1280
N_CLASSES    = 10
HEAD_EPOCHS  = 10
HEAD_BATCH   = 256
HEAD_LR      = 1e-3

# ArrayType(FloatType()) columns are stored as plain lists in Parquet,
# so pyarrow can read them directly without VectorUDT struct unpacking.
(
    features_train
    .select('mobilenet_features', 'label')
    .write.mode('overwrite').parquet(FASHION_TRAIN_PARQUET)
)
(
    features_test
    .select('mobilenet_features', 'label')
    .write.mode('overwrite').parquet(FASHION_TEST_PARQUET)
)
print('Features written to Parquet.')

In [ ]:
class FashionHead(nn.Module):
    """Lightweight classification head trained on top of frozen MobileNetV2 features."""
    def __init__(self, in_dim=FEATURE_DIM, n_classes=N_CLASSES, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, n_classes),
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
def train_fashion_head():
    """
    Self-contained training function for TorchDistributor.
    All imports and class definitions are repeated inside — TorchDistributor
    pickles this function and runs it in an isolated worker process.
    """
    import torch, torch.nn as nn, torch.nn.functional as F
    import torch.optim as optim
    import pyarrow.parquet as pq
    from torch.utils.data import TensorDataset, DataLoader

    def load_parquet(path):
        table = pq.read_table(path)
        X = torch.tensor(table['mobilenet_features'].to_pylist(), dtype=torch.float32)
        y = torch.tensor(table['label'].to_pylist(), dtype=torch.long)
        return TensorDataset(X, y)

    class FashionHead(torch.nn.Module):
        def __init__(self, in_dim=1280, n_classes=10, dropout=0.3):
            super().__init__()
            self.net = torch.nn.Sequential(
                torch.nn.Linear(in_dim, 256), torch.nn.ReLU(),
                torch.nn.Dropout(dropout),    torch.nn.Linear(256, n_classes),
            )
        def forward(self, x): return self.net(x)

    device       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    train_loader = DataLoader(load_parquet(FASHION_TRAIN_PARQUET),
                             batch_size=HEAD_BATCH, shuffle=True)
    test_loader  = DataLoader(load_parquet(FASHION_TEST_PARQUET),
                             batch_size=HEAD_BATCH)

    model = FashionHead().to(device)
    opt   = optim.Adam(model.parameters(), lr=HEAD_LR)

    for epoch in range(1, HEAD_EPOCHS + 1):
        model.train()
        total_loss = 0.0
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            opt.zero_grad()
            loss = F.cross_entropy(model(X), y)
            loss.backward()
            opt.step()
            total_loss += loss.item()
        if epoch % 2 == 0 or epoch == 1:
            print(f'  Epoch {epoch:2d}/{HEAD_EPOCHS}  '
                  f'loss={total_loss / len(train_loader):.4f}')

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for X, y in test_loader:
            X, y = X.to(device), y.to(device)
            correct += (model(X).argmax(1) == y).sum().item()
            total   += y.size(0)
    print(f'FashionHead test accuracy: {correct / total:.4f}')

    return model.state_dict()


fashion_state_dict = TorchDistributor(
    num_processes=1, local_mode=True, use_gpu=False,
).run(train_fashion_head)

fashion_model = FashionHead()
fashion_model.load_state_dict(fashion_state_dict)
fashion_model.eval()
print('FashionHead reconstructed on driver.')

<a class="anchor" id="4.3">

## **4.3. Distributed Inference**
</a>

With the head trained, we run it across the full test feature set in a distributed, Spark-native way using two complementary mechanisms:

1. **`spark.sparkContext.broadcast()`** — serialises the `state_dict` bytes once and ships them to every worker node, where they are cached in memory. This is far more efficient than re-serialising inside every task. We also broadcast the `CLASS_NAMES` dictionary, demonstrating that broadcast variables can carry any Python object.
2. **`predict_batch_udf`** — the setup function `make_head_inference_fn` deserialises the broadcast bytes on the worker and reconstructs the `FashionHead`. The inner `predict` function runs forward passes on batches of 1280-dimensional feature vectors and returns the predicted class index per row.

In [ ]:
# Serialise trained weights to bytes and broadcast to all workers
_buf = io.BytesIO()
torch.save(fashion_state_dict, _buf)
fashion_weights_bc = spark.sparkContext.broadcast(_buf.getvalue())

# Broadcast class-name lookup (demonstrates general-purpose broadcast usage)
class_names_bc = spark.sparkContext.broadcast(
    {i: name for i, name in enumerate(CLASS_NAMES)}
)

print(f'Weights broadcast ID   : {fashion_weights_bc.id}')
print(f'Class names broadcast ID: {class_names_bc.id}')
print(f'Serialised model size  : {len(_buf.getvalue()) / 1024:.1f} KB')

In [ ]:
def make_head_inference_fn():
    """Setup function for predict_batch_udf — called once per worker."""
    import torch, io, numpy as np

    class FashionHead(torch.nn.Module):
        def __init__(self, in_dim=1280, n_classes=10, dropout=0.3):
            super().__init__()
            self.net = torch.nn.Sequential(
                torch.nn.Linear(in_dim, 256), torch.nn.ReLU(),
                torch.nn.Dropout(dropout),    torch.nn.Linear(256, n_classes),
            )
        def forward(self, x): return self.net(x)

    state_dict = torch.load(
        io.BytesIO(fashion_weights_bc.value), weights_only=True
    )
    model = FashionHead()
    model.load_state_dict(state_dict)
    model.eval()

    def predict(features_batch: np.ndarray) -> np.ndarray:
        """features_batch: (N, 1280) → (N,) float64 predicted class indices."""
        with torch.no_grad():
            logits = model(torch.from_numpy(features_batch.astype(np.float32)))
        return logits.argmax(dim=1).double().numpy()

    return predict


head_inference_udf = predict_batch_udf(
    make_head_inference_fn,
    return_type=DoubleType(),
    batch_size=256,
)

# Apply inference UDF and attach human-readable class names via the broadcast lookup
label_to_name_udf = F.udf(
    lambda lbl: class_names_bc.value.get(int(lbl), 'Unknown'), StringType()
)

predictions_dl = (
    features_test
    .withColumn('prediction',      head_inference_udf(col('mobilenet_features')))
    .withColumn('actual_class',    label_to_name_udf(col('label')))
    .withColumn('predicted_class', label_to_name_udf(col('prediction')))
)

predictions_dl.select('label', 'prediction', 'actual_class', 'predicted_class')\
    .show(10, truncate=30)

<a class="anchor" id="4.4">

## **4.4. Evaluation**
</a>

In [ ]:
results_dl = evaluate_multiclass(
    predictions_dl,
    'MobileNetV2 + FashionHead (Transfer Learning)'
)

In [ ]:
cm_dl = plot_confusion_matrix(
    predictions_dl,
    'MobileNetV2 + FashionHead (Transfer Learning)'
)

### Results

The transfer-learning pipeline typically achieves accuracy in the **~88–92%** range, outperforming the flat MLPC by a few percentage points. The improvement comes entirely from the pretrained convolutional features: MobileNetV2's depthwise-separable convolutions encode spatial patterns (edges, textures, shapes) that a flat 784-input MLP cannot capture.

The classification head itself is simple (256 units, 10 epochs, no hyperparameter search); the heavy lifting is done by the frozen backbone. The `TorchDistributor` + `predict_batch_udf` + broadcast pattern demonstrated here scales to multi-node Spark clusters without code changes: `TorchDistributor` handles distributed training via DDP, `predict_batch_udf` distributes inference across executors, and `broadcast` ensures model weights are shipped to workers exactly once per job rather than once per task.

<a class="anchor" id="5">

# **5. Saving the Pipelines**
</a>

**Why save the models?** Persisting trained models gives several practical advantages:

- **No retraining at inference time.** Loading a saved model is orders of magnitude cheaper than refitting the full pipeline — especially important for the MLPC, which ran a 6-fold cross-validation.
- **Reproducibility.** The exact scaler fit (min/max values per pixel dimension) and model weights are frozen at the point of saving, eliminating train/serve skew.
- **Portability.** A saved Spark `PipelineModel` can be loaded by any downstream Spark job — a batch scoring script, a Structured Streaming application, or another notebook — without access to the training code.

We save the Spark MLPC pipeline with the standard `PipelineModel.write()` API and the PyTorch `FashionHead` with `torch.save()`, keeping each artefact in the format native to its framework.

In [ ]:
# --- Save the Spark MLPC PipelineModel ---
# Persists every stage: MinMaxScaler (with fitted min/max per feature) + MLPC weights.
pipeline_model_mlpc.write().overwrite().save('./outputs/fashion_pipeline_mlpc')
print('Spark MLPC pipeline saved to ./outputs/fashion_pipeline_mlpc')

# --- Save the PyTorch FashionHead weights ---
# torch.save stores the state_dict as a .pt file, readable by any PyTorch environment.
torch.save(fashion_state_dict, './outputs/fashion_head_weights.pt')
print('FashionHead weights saved to ./outputs/fashion_head_weights.pt')

### Loading the models back

The cells below demonstrate how to reload each artefact from disk, confirming the save/load round-trip works correctly and the predictions are identical to those produced during training.

In [ ]:
from pyspark.ml import PipelineModel

# --- Reload Spark MLPC pipeline ---
loaded_mlpc_pipeline = PipelineModel.load('./fashion_pipeline_mlpc')
reloaded_preds = loaded_mlpc_pipeline.transform(df_test_vec)
reloaded_acc = MulticlassClassificationEvaluator(
    labelCol='label', predictionCol='prediction', metricName='accuracy'
).evaluate(reloaded_preds)
print(f'Reloaded MLPC accuracy : {reloaded_acc:.4f}  (should match original)')

# --- Reload PyTorch FashionHead ---
loaded_state = torch.load('./fashion_head_weights.pt', weights_only=True)
loaded_head  = FashionHead()
loaded_head.load_state_dict(loaded_state)
loaded_head.eval()
print('FashionHead reloaded from disk successfully.')

<a class="anchor" id="6">

# **6. Comparing Results**
</a>

Having trained and evaluated both models on the same held-out test set, we now compare them side by side. Any difference in accuracy is attributable **solely to the model architecture** — both receive identical label columns, are evaluated with the same `MulticlassClassificationEvaluator`, and never see the test set during training.

In [ ]:
results_table = pd.DataFrame([results_mlpc, results_dl])
results_table = results_table.set_index('model').round(4)

print('=== Model Comparison — Test Set Metrics ===')
print(results_table.to_string())
print()

best_model = results_table['Accuracy'].idxmax()
best_acc   = results_table.loc[best_model, 'Accuracy']
print(f'\u2605  Best model by Accuracy: {best_model}  (Accuracy = {best_acc:.4f})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
colors = ['#4C72B0', '#DD8452']
short_names = ['Spark\nMLPC', 'MobileNetV2\n+ Head']

for ax, metric in zip(axes, ['Accuracy', 'F1']):
    vals = results_table[metric].values
    bars = ax.bar(short_names, vals, color=colors, width=0.5, edgecolor='white')
    ax.set_ylim(max(0, min(vals) - 0.05), min(1.0, max(vals) + 0.05))
    ax.set_title(metric, fontsize=12)
    ax.set_ylabel(metric)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
                f'{v:.4f}', ha='center', va='bottom', fontsize=10)

plt.suptitle('Model Comparison — Fashion MNIST Test Set', fontsize=13)
plt.tight_layout()
plt.show()

### Key Takeaways

**Spark MLPC** demonstrates that PySpark has a first-class, distributed neural network estimator. It requires zero external dependencies, fits natively into any Spark ML `Pipeline`, and supports standard hyperparameter tuning via `CrossValidator` — making it the right choice when portability and pipeline integration matter more than state-of-the-art accuracy.

**MobileNetV2 + Transfer Learning** shows that Spark can act as the **orchestration and distribution layer** for any PyTorch model. `predict_batch_udf` handles distributed inference efficiently (model loaded once per worker), `TorchDistributor` enables scale-out training, and `broadcast` ensures large model artefacts are sent to workers exactly once per job. This pattern scales to any model size or dataset size without changes to the Spark-facing code.

The accuracy gap between the two approaches (typically ~3–6 pp) reflects the value of pretrained convolutional features for image tasks — not a limitation of Spark's native neural network support.